# Notebook 03 — Manifold Visualization


---

## Goals
1. Load trained weights and feature data
2. Visualize the raw input space: emotions tangled together
3. Visualize the latent hidden space: emotions untangled by the network
4. Apply t-SNE for an even cleaner cluster view
5. Run a sensitivity check, which neurons react most to each emotion

---

### Core Idea
The neural network reshapes space layer by layer — like uncrumpling a ball of paper.
By the final hidden layer, emotions that were mathematically tangled in 52-dimensional
MFCC space should now live in clearly separated regions.
This notebook makes that visible.

In [ ]:
#Imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import sys

sys.path.append('../src')
from numpy_network import NeuralNetwork

print('Imports successful.')

---
## Cell 1 : Load Data and Trained Weights

In [ ]:
# Load feature matrix and labels
X      = np.load('../saved_models/X.npy')       # shape: (52, n_samples)
y      = np.load('../saved_models/y.npy')       # shape: (4,  n_samples)
labels = np.load('../saved_models/labels.npy')  # shape: (n_samples,)

print(f'X shape      : {X.shape}')
print(f'y shape      : {y.shape}')
print(f'labels shape : {labels.shape}')
print(f'Unique labels: {np.unique(labels)}')

In [ ]:
# Rebuild network architecture and load saved weights
nn = NeuralNetwork(layer_sizes=[52, 64, 32, 4])
nn.load('../saved_models/trained_weights.npz')

print('Network loaded successfully.')

---
## Cell 2 : Raw Input Space (Before the Network)

This is the **crumpled ball** — 52-dimensional MFCC features compressed to 2D with PCA.
Expect a messy cloud where emotion clusters overlap heavily.

In [ ]:
# PCA on raw input features
# X.T → (n_samples, 52) because PCA expects (samples, features)
pca   = PCA(n_components=2)
X_2d  = pca.fit_transform(X.T)

print(f'Explained variance by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Plot
colors = {'neutral': 'gray', 'happy': 'gold', 'sad': 'royalblue', 'angry': 'crimson'}

plt.figure(figsize=(8, 6))
for emotion, color in colors.items():
    mask = labels == emotion
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=color, label=emotion, alpha=0.6, s=40, edgecolors='none')

plt.title('RAW INPUT SPACE :Emotions Tangled Together', fontsize=13, fontweight='bold')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../saved_models/plot_raw_input_space.png', dpi=150)
plt.show()
print('Saved: plot_raw_input_space.png')

---
## Cell 3 : Latent Hidden Space (After the Network Untangles)

We extract activations from the **last hidden layer** (layer −2, just before the output).
This is the space the network has learned to reshape the data into.
If training succeeded, emotion clusters should now be visibly separated.

In [ ]:
# Get last hidden layer activations
# layer=-2 means: last hidden layer, one step before the output
hidden = nn.get_hidden_representation(X, layer=-2)  # shape: (32, n_samples)
print(f'Hidden representation shape: {hidden.shape}')

# PCA on hidden representations
pca_hidden   = PCA(n_components=2)
hidden_2d    = pca_hidden.fit_transform(hidden.T)   # (n_samples, 2)

print(f'Explained variance by 2 PCs: {pca_hidden.explained_variance_ratio_.sum()*100:.1f}%')

# Plot
plt.figure(figsize=(8, 6))
for emotion, color in colors.items():
    mask = labels == emotion
    plt.scatter(hidden_2d[mask, 0], hidden_2d[mask, 1],
                c=color, label=emotion, alpha=0.6, s=40, edgecolors='none')

plt.title('LATENT SPACE — Emotion Manifolds Untangled by the Network', fontsize=13, fontweight='bold')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../saved_models/plot_latent_space_pca.png', dpi=150)
plt.show()
print('Saved: plot_latent_space_pca.png')

---
## Cell 4 : t-SNE Visualization of Latent Space

PCA is linear — it can only rotate and scale the space.
**t-SNE** is non-linear and much better at revealing cluster structure.
This usually gives the clearest picture of whether the manifolds are truly separated.

In [ ]:
# t-SNE on hidden representations
# perplexity controls how many neighbours each point considers — try 20, 30, 50
tsne         = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
hidden_tsne  = tsne.fit_transform(hidden.T)   # (n_samples, 2)

# Plot
plt.figure(figsize=(8, 6))
for emotion, color in colors.items():
    mask = labels == emotion
    plt.scatter(hidden_tsne[mask, 0], hidden_tsne[mask, 1],
                c=color, label=emotion, alpha=0.6, s=40, edgecolors='none')

plt.title('t-SNE of Latent Space — Emotion Clusters', fontsize=13, fontweight='bold')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../saved_models/plot_latent_space_tsne.png', dpi=150)
plt.show()
print('Saved: plot_latent_space_tsne.png')

---
## Cell 5 : Sensitivity Check

**Which neurons in the hidden layer react most strongly to each emotion?**

For each emotion, we compute the average activation across all 32 hidden neurons.
The top-3 most active neurons reveal which directions in latent space
the network has learned to associate with each emotional signal.

In [ ]:
# Re-run forward pass to get fresh hidden activations
nn.forward(X)
hidden_acts = nn.activations[-2]   # shape: (32, n_samples)

print('Average neuron activation per emotion (top 3 most active neurons):')
print('-' * 60)
for emotion in ['neutral', 'happy', 'sad', 'angry']:
    mask      = labels == emotion
    avg       = np.mean(hidden_acts[:, mask], axis=1)   # mean over samples of this emotion
    top3_idx  = np.argsort(avg)[-3:][::-1]              # top 3 neuron indices
    top3_vals = avg[top3_idx].round(4)
    print(f'  {emotion:8s} | top neurons: {top3_idx} | activations: {top3_vals}')

In [ ]:
# Heatmap: average activation of every hidden neuron for every emotion
emotion_list = ['neutral', 'happy', 'sad', 'angry']
heatmap_data = np.zeros((len(emotion_list), hidden_acts.shape[0]))  # (4, 32)

for i, emotion in enumerate(emotion_list):
    mask = labels == emotion
    heatmap_data[i] = np.mean(hidden_acts[:, mask], axis=1)

plt.figure(figsize=(14, 4))
plt.imshow(heatmap_data, aspect='auto', cmap='RdYlGn', interpolation='nearest')
plt.colorbar(label='Mean Activation')
plt.yticks(range(len(emotion_list)), emotion_list)
plt.xlabel('Hidden Neuron Index')
plt.title('Neuron Sensitivity Heatmap — Which Neurons Fire for Which Emotion?',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../saved_models/plot_sensitivity_heatmap.png', dpi=150)
plt.show()
print('Saved: plot_sensitivity_heatmap.png')

---
## Summary of Outputs

| Plot | File | What it shows |
|---|---|---|
| Raw Input Space | `plot_raw_input_space.png` | Emotions tangled in MFCC space |
| Latent Space PCA | `plot_latent_space_pca.png` | Manifolds after network untangling |
| Latent Space t-SNE | `plot_latent_space_tsne.png` | Clearest cluster separation |
| Sensitivity Heatmap | `plot_sensitivity_heatmap.png` | Which neurons respond to which emotion |

---

> **If your network trained well**, the latent space plots will show clearly separated
> coloured clusters — visual proof that the neural network has successfully
> *untangled the emotion manifolds* from the raw audio feature space.